In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# CELL 0 — Shared prep config  (run once; used by ALL 3 datasets)

In [2]:

import os, re, random, hashlib, json
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
Image.MAX_IMAGE_PIXELS = None

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# ---- shared output contract (identical for every dataset) ----
TARGET_SIZE  = 256                       # materialize letterboxed to 256²; online random-crop to 224² at train time
PAD_COLOR    = (0, 0, 0)                  # letterbox fill
IMG_EXTS     = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
OUT_ROOT     = Path("/kaggle/working/prepared")   # cleaned dataset built here, then published as a Kaggle Dataset
SPLIT_RATIOS = {"train": 0.80, "val": 0.10, "test": 0.10}   # used when a source ships no split (rice)

# the manifest schema every dataset must emit, so the three merge cleanly:
MANIFEST_COLS = ["src_path", "filename", "label", "source_dataset", "split", "group_id"]

INPUT_ROOT = "/kaggle/input"
print("Mounted datasets:", os.listdir(INPUT_ROOT))
print("Output root     :", OUT_ROOT)

Mounted datasets: ['datasets']
Output root     : /kaggle/working/prepared


# CELL R1 — Rice Set 1: locate dataset + build raw manifest

In [3]:
cands = [os.path.join(INPUT_ROOT, d) for d in os.listdir(INPUT_ROOT)]
DATA_ROOT = next((c for c in cands if "rice" in c.lower()), cands[0])
print("Rice S1 DATA_ROOT =", DATA_ROOT)

rows = []
for p in Path(DATA_ROOT).rglob("*"):
    if p.is_file() and p.suffix.lower() in IMG_EXTS:
        rows.append({"src_path": str(p), "filename": p.name,
                     "raw_label": p.parent.name, "split": "all"})

s1 = pd.DataFrame(rows)
print("Total images:", len(s1), "(expect 5932)")
print("\nRaw labels:\n", s1["raw_label"].value_counts())
# batch-prefix peek — confirms the burst-capture pattern behind the near-dups
s1["prefix"] = s1["filename"].str.extract(r"^([A-Za-z]+\d?)")
print("\nTop filename prefixes (burst-capture signal):")
print(s1["prefix"].value_counts().head(10))

Rice S1 DATA_ROOT = /kaggle/input/datasets
Total images: 5932 (expect 5932)

Raw labels:
 raw_label
Brownspot          1600
Bacterialblight    1584
Blast              1440
Tungro             1308
Name: count, dtype: int64

Top filename prefixes (burst-capture signal):
prefix
TUNGRO2             277
TUNGRO4             277
TUNGRO3             277
TUNGRO1             277
BACTERIALBLIGHT     264
BACTERIALBLIGHT1    264
BACTERIALBLIGHT2    264
BACTERAILBLIGHT5    264
BACTERAILBLIGHT4    264
BACTERAILBLIGHT3    264
Name: count, dtype: int64


# CELL R2 — Rice Set 1: normalize labels to shared canonical names + crop prefix

In [4]:
# Explicit map so Set 1 and Set 2 use IDENTICAL keys when merged in Notebook D.
RICE_LABEL_MAP = {
    "bacterialblight": "bacterial_blight",
    "brownspot":       "brown_spot",
    "blast":           "blast",
    "tungro":          "tungro",
    # Set 2 names (kept here as the single source of truth for both notebooks):
    "bacterial leaf blight": "bacterial_blight",
    "brown spot":            "brown_spot",
    "leaf smut":             "leaf_smut",
}

def rice_key(raw):
    k = raw.strip().lower()
    if k not in RICE_LABEL_MAP:
        raise KeyError(f"Unmapped rice label: {raw!r} — add it to RICE_LABEL_MAP")
    return RICE_LABEL_MAP[k]

s1["key"]            = s1["raw_label"].map(rice_key)
s1["label"]         = "rice__" + s1["key"]
s1["source_dataset"] = "rice_s1_nirmalsankalana"

print("Canonical rice classes:", sorted(s1["key"].unique()))
print("\nPer-class counts:")
print(s1["label"].value_counts().sort_index())

Canonical rice classes: ['bacterial_blight', 'blast', 'brown_spot', 'tungro']

Per-class counts:
label
rice__bacterial_blight    1584
rice__blast               1440
rice__brown_spot          1600
rice__tungro              1308
Name: count, dtype: int64


# CELL R3 — Rice S1: exact + perceptual hashes  (slow pass, run once)

In [5]:
import hashlib

def _dct_matrix(N):
    n = np.arange(N); k = n.reshape(-1, 1)
    M = np.sqrt(2.0 / N) * np.cos(np.pi * (2 * n + 1) * k / (2 * N))
    M[0, :] /= np.sqrt(2.0); return M
_D32 = _dct_matrix(32)

def phash64(pil_img):
    g = pil_img.convert("L").resize((32, 32), Image.BILINEAR)
    d = _D32 @ np.asarray(g, dtype=np.float64) @ _D32.T
    flat = d[:8, :8].flatten(); med = np.median(flat[1:])
    h = np.uint64(0)
    for b in (flat > med):
        h = (h << np.uint64(1)) | np.uint64(bool(b))
    return h

file_md5, phash, bad = [], [], []
for i, p in enumerate(s1["src_path"]):
    try:
        with open(p, "rb") as f: file_md5.append(hashlib.md5(f.read()).hexdigest())
        with Image.open(p) as im: phash.append(int(phash64(im)))
    except Exception as e:
        file_md5.append(None); phash.append(None); bad.append((p, str(e)))
    if (i + 1) % 1500 == 0: print(f"  hashed {i+1}/{len(s1)}")

s1["file_md5"] = file_md5; s1["phash"] = phash
print("Done. Unreadable:", len(bad))
print("Exact byte-duplicate files:", int(s1['file_md5'].duplicated(keep=False).sum()),
      "across", s1['file_md5'].nunique(), "unique blobs")

  hashed 1500/5932
  hashed 3000/5932
  hashed 4500/5932
Done. Unreadable: 0
Exact byte-duplicate files: 2234 across 4794 unique blobs


# CELL R4 — Rice S1: near-dup clustering + TRUE unique-count diagnostic

In [6]:
PHASH_THRESH = 6

valid = s1.dropna(subset=["phash"]).copy().reset_index()
H = valid["phash"].to_numpy(dtype=np.uint64); N = len(H)

_c1=np.uint64(0x5555555555555555); _c2=np.uint64(0x3333333333333333)
_c4=np.uint64(0x0f0f0f0f0f0f0f0f); _cm=np.uint64(0x0101010101010101)
def popcount64(x):
    x = x - ((x >> np.uint64(1)) & _c1)
    x = (x & _c2) + ((x >> np.uint64(2)) & _c2)
    x = (x + (x >> np.uint64(4))) & _c4
    return (x * _cm >> np.uint64(56)) & np.uint64(0x7f)

parent = list(range(N))
def find(a):
    while parent[a] != a: parent[a] = parent[parent[a]]; a = parent[a]
    return a
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb: parent[rb] = ra

BLK = 500
for s in range(0, N, BLK):
    e = min(s + BLK, N)
    dist = popcount64(np.bitwise_xor(H[s:e, None], H[None, :]))
    for r in range(e - s):
        for j in np.where(dist[r] <= PHASH_THRESH)[0]:
            if j > s + r: union(s + r, int(j))
    if (e % 2500) < BLK: print(f"  clustered {e}/{N}")

valid["group_id"] = [f"s1_g{find(i):05d}" for i in range(N)]
s1 = s1.merge(valid.set_index("index")[["group_id"]], left_index=True, right_index=True, how="left")
solo = s1["group_id"].isna()
s1.loc[solo, "group_id"] = [f"s1_solo{i:05d}" for i in range(int(solo.sum()))]

g = s1.groupby("group_id"); sizes = g.size()
print("Total images        :", len(s1))
print("Unique groups        :", s1['group_id'].nunique(), "  <-- TRUE effective size")
print("Multi-image groups   :", int((sizes > 1).sum()), "| largest cluster:", int(sizes.max()))
print("\nCluster-size distribution:")
print(sizes.value_counts().sort_index().head(15))
print("\nTRUE unique images per class (groups counted once):")
print(s1.groupby('key')['group_id'].nunique().sort_index())
print("\nGroups spanning >1 label (probable mislabels):", int((g['key'].nunique() > 1).sum()))

  clustered 2500/5932
  clustered 5000/5932
Total images        : 5932
Unique groups        : 2066   <-- TRUE effective size
Multi-image groups   : 1816 | largest cluster: 13

Cluster-size distribution:
1     250
2     732
3     587
4     333
5      42
6      56
7       8
8      21
9      27
10      6
13      4
Name: count, dtype: int64

TRUE unique images per class (groups counted once):
key
bacterial_blight    514
blast               477
brown_spot          606
tungro              469
Name: group_id, dtype: int64

Groups spanning >1 label (probable mislabels): 0


# CELL R5 — Rice S1: thin each near-dup group to its sharpest frame

In [7]:
from numpy.lib.stride_tricks import sliding_window_view as swv
def lap_var(gray):
    a = np.asarray(gray, dtype=np.float64)
    k = np.array([[0,1,0],[1,-4,1],[0,1,0]], dtype=np.float64)
    w = swv(a, (3,3))
    return float((w * k).sum(axis=(2,3)).var())

sharp = []
for i, p in enumerate(s1["src_path"]):
    try:
        with Image.open(p) as im:
            sharp.append(lap_var(im.convert("L").resize((128,128), Image.BILINEAR)))
    except Exception:
        sharp.append(-1.0)
    if (i + 1) % 1500 == 0: print(f"  scored {i+1}/{len(s1)}")
s1["sharpness"] = sharp

# keep the single sharpest image in each near-dup group
s1_thin = (s1.sort_values("sharpness", ascending=False)
             .drop_duplicates("group_id", keep="first").copy())

print("Before thinning:", len(s1), "| after:", len(s1_thin))
print("\nPer-class after thinning:")
print(s1_thin["label"].value_counts().sort_index())
vc = s1_thin["label"].value_counts()
print("\nImbalance ratio:", round(vc.max() / vc.min(), 2))

  scored 1500/5932
  scored 3000/5932
  scored 4500/5932
Before thinning: 5932 | after: 2066

Per-class after thinning:
label
rice__bacterial_blight    514
rice__blast               477
rice__brown_spot          606
rice__tungro              469
Name: count, dtype: int64

Imbalance ratio: 1.29


# CELL R6 — Rice S1: letterbox to 256², write images + manifest

In [8]:
OUT_IMG = OUT_ROOT / "rice_s1" / "images"
OUT_IMG.mkdir(parents=True, exist_ok=True)
BLANK_VAR_THRESH = 8.0

records, dropped_blank, n_fail = [], 0, 0
for i, r in enumerate(s1_thin.itertuples(index=False)):
    try:
        if r.sharpness < BLANK_VAR_THRESH:
            dropped_blank += 1; continue
        with Image.open(r.src_path) as im:
            im = ImageOps.exif_transpose(im).convert("RGB")           # flattens the 156 RGBA files
            im = ImageOps.pad(im, (TARGET_SIZE, TARGET_SIZE),
                              method=Image.BILINEAR, color=PAD_COLOR)  # kills the dimension shortcut
            fn = f"{r.key}_{i:05d}.jpg"
            im.save(OUT_IMG / fn, "JPEG", quality=92)
        records.append({"src_path": r.src_path, "filename": fn, "label": r.label,
                        "source_dataset": "rice_s1_nirmalsankalana", "split": "unassigned",
                        "group_id": r.group_id})
    except Exception:
        n_fail += 1

man = pd.DataFrame(records, columns=MANIFEST_COLS)
man.to_csv(OUT_ROOT / "rice_s1" / "manifest.csv", index=False)
print("Near-blank dropped:", dropped_blank, "| failed:", n_fail, "| written:", len(man))
print("\nFinal per-class:")
print(man["label"].value_counts().sort_index())

Near-blank dropped: 0 | failed: 0 | written: 2066

Final per-class:
label
rice__bacterial_blight    514
rice__blast               477
rice__brown_spot          606
rice__tungro              469
Name: count, dtype: int64
